# Trabajo Práctico Integrador - Introducción al Análisis de Datos
## ***Segunda Parte***: Preparación y Transformación de Datos

**Caso:** análisis y predicción de cancelaciones en reservas hoteleras.

> La empresa busca comprender qué factores influyen en la cancelación de reservas en hoteles urbanos y resort, analizando variables como fechas de estadía, tipo de cliente, canal de reserva, historial previo y tarifas, con el fin de identificar patrones y mejorar la gestión operativa.




## Desarrollado por

- **Integrantes:** `Matias Carro` y `Hugo Catalan`
- **Comisión:** 11
- **Dataset asignado:** Dataset K
- **Entrega:** Primer Entrega - Semana 3
- **Fecha:** 

## Importación de librerías

### Librerias a utilizar:

- **pandas:** para cargar el dataset y trabajar con datos en forma de tablas (DataFrames).
- **numpy:** para realizar operaciones numéricas y manejar arreglos de forma eficiente.



In [35]:
import pandas as pd
import numpy as np

print("Versión de Pandas:", pd.__version__)
print("Versión de numpy:", np.__version__)
print("\nLibrerías cargadas correctamente.")


Versión de Pandas: 3.0.5
Versión de numpy: 2.5.2

Librerías cargadas correctamente.


## 1. Presentacion del problema

El caso de estudio se centra en el análisis de reservas hoteleras con el objetivo de comprender qué factores están asociados a la cancelación de estadías. El dataset asignado contiene información detallada de cada reserva, incluyendo tipo de hotel, fechas de llegada, duración de la estadía, composición del grupo, país de origen, canal de reserva, tipo de cliente, historial previo, tarifa promedio por noche y características operativas como depósito, agente, pedidos especiales y cambios realizados.


### Relación entre datos, información y conocimiento
En este trabajo partimos de los **datos** que son valores crudos del sistema de reservas: fechas, cantidades, categorías y códigos.  
Mediante el análisis exploratorio estos datos se convierten en **información**, como distribuciones, patrones y diferencias entre reservas canceladas y no canceladas.  
A partir de esa información generamos el **conocimiento** que nos permite entender el comportamiento de los clientes y detectar factores que podrían influir en la cancelación de una reserva.  

Esta relación es clave para el caso: los datos del hotel por sí solos no dicen nada, pero al transformarlos en información y luego interpretarlos, podemos identificar variables relevantes (como `lead_time`, `deposit_type` o `customer_type`) que ayudan a explicar por qué algunas reservas se cancelan y otras no.

### Ciclo de vida del análisis
Este trabajo se enmarca en el ciclo de vida del análisis de datos, que incluye:
1. Obtención del dataset asignado.  
2. Comprensión inicial del problema (cancelaciones hoteleras).  
3. Exploración y limpieza mínima (EDA).  
4. Transformación y preparación de variables relevantes.  
5. Interpretación y comunicación de resultados.

### Variable objetivo
La **variable objetivo** del análisis es **`is_canceled`**, que indica si la reserva fue cancelada (`1`) o no (`0`).  
Su distribución será calculada y analizada en las próximas secciones para comprender el comportamiento general del conjunto de datos y orientar las preguntas del análisis.

### Preguntas iniciales que orientan el trabajo
- ¿Qué características diferencian a las reservas canceladas de las no canceladas?  
- ¿Influyen el tipo de hotel o el canal de reserva en la cancelación?  
- ¿Las reservas con mayor anticipación (`lead_time`) presentan mayor probabilidad de cancelación?  
- ¿Los clientes con pedidos especiales o estacionamiento tienden a cancelar menos?  
- ¿Las políticas de depósito (`deposit_type`) reducen la cancelación?  
- ¿Existen segmentos de mercado con mayor riesgo de cancelación?  


---

## 2. Dataset Asignado

- **Comisión:** 11
- **Dataset asignado:** Dataset K


---

## 3. Carga del dataset

Se carga el Dataset perteneciente a la comisión 11:


In [36]:
df = pd.read_csv("hotel booking TPI grupo K.csv")

print("Dataset cargado correctamente.")


Dataset cargado correctamente.


---

## 4. Resumen del dataset

Resumen de los datos 

In [37]:
filas, columnas = df.shape
print(f"El dataset tiene {filas} filas y {columnas} columnas.")

print("Columnas del dataset:")
print(df.columns.tolist())

print("\nTipos de datos:")
print(df.info())


El dataset tiene 25000 filas y 32 columnas.
Columnas del dataset:
['booking_id', 'hotel', 'is_canceled', 'lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'arrival_date', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'meal', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'reserved_room_type', 'assigned_room_type', 'booking_changes', 'deposit_type', 'agent', 'company', 'days_in_waiting_list', 'customer_type', 'adr', 'required_car_parking_spaces', 'total_of_special_requests']

Tipos de datos:
<class 'pandas.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 32 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   booking_id                      25000 non-null  str    
 1   hotel                          

El dataset tiene 25.000 filas y 32 columnas.  
Las variables incluyen información temporal, categórica y numérica relevante para el análisis.


### Diccionario de variables 

| Variable | Traducción | Representación | Tipo de Datos |
|----------|------------|----------------|-----------|
| booking_id | ID de reserva | Identificador único de cada reserva. | str |
| hotel | Tipo de hotel | Indica si la reserva corresponde a City Hotel (Ciudad) o Resort Hotel (Resort). | str |
| is_canceled | Cancelada | Indica si la reserva fue cancelada (1) o no (0). | int64 |
| lead_time | Anticipación | Días entre la fecha de reserva y la fecha de llegada. | int64 |
| arrival_date_year | Año de llegada | Año en el que el huésped llega al hotel. | int64 |
| arrival_date_month | Mes de llegada | Mes en el que el huésped llega al hotel. | str |
| arrival_date_week_number | Semana de llegada | Número de semana del año en la que llega el huésped. | int64 |
| arrival_date_day_of_month | Día del mes de llegada | Día del mes en el que llega el huésped. | int64 |
| arrival_date | Fecha de llegada | Fecha completa de llegada (YYYY-MM-DD). | str |
| stays_in_weekend_nights | Noches de fin de semana | Cantidad de noches en fines de semana. | int64 |
| stays_in_week_nights | Noches de semana | Cantidad de noches de lunes a jueves. | int64 |
| adults | Adultos | Número de adultos en la reserva. | int64 |
| children | Niños | Número de niños en la reserva. | float64 |
| babies | Bebés | Número de bebés en la reserva. | int64 |
| meal | Tipo de comida | Plan de comidas asociado a la reserva (BB, HB, SC, etc.). | str |
| country | País | País de origen del huésped. | str |
| market_segment | Segmento de mercado | Tipo de cliente según el canal de adquisición. | str |
| distribution_channel | Canal de distribución | Canal por el cual se realizó la reserva. | str |
| is_repeated_guest | Huésped repetido | Indica si el cliente ya se alojó anteriormente. | int64 |
| previous_cancellations | Cancelaciones previas | Cantidad de reservas previas canceladas por el cliente. | int64 |
| previous_bookings_not_canceled | Reservas previas no canceladas | Cantidad de reservas previas completadas por el cliente. | int64 |
| reserved_room_type | Habitación reservada | Tipo de habitación solicitada originalmente. | str |
| assigned_room_type | Habitación asignada | Tipo de habitación finalmente asignada. | str |
| booking_changes | Cambios en la reserva | Número de modificaciones realizadas a la reserva. | int64 |
| deposit_type | Tipo de depósito | Política de depósito aplicada. | str |
| agent | Agente | Código del agente que gestionó la reserva. | float64 |
| company | Compañía | Código de la empresa asociada a la reserva. | float64 |
| days_in_waiting_list | Días en lista de espera | Tiempo que la reserva permaneció en espera antes de confirmarse. | int64 |
| customer_type | Tipo de cliente | Clasificación del cliente (Transient, Contract, Group, etc.). | str |
| adr | Tarifa promedio diaria | Precio promedio por noche de la reserva. | float64 |
| required_car_parking_spaces | Estacionamiento requerido | Cantidad de espacios de estacionamiento solicitados. | int64 |
| total_of_special_requests | Pedidos especiales | Número de solicitudes especiales realizadas por el cliente. | int64 |




---

## 5. Limpieza de los Datos

Se comienza la limpieza de los datos

### 0. Preparativos

Se crea una copia del dataset original para conservar un respaldo íntegro de los datos. A partir de este punto, todas las modificaciones se realizarán exclusivamente sobre la copia de trabajo.

El dataset original permanece sin alteraciones y funciona como *referencia histórica* ante cualquier revisión o necesidad de volver al estado inicial.

In [38]:
df_original = df.copy()
print("Copia del dataset creada correctamente.")

Copia del dataset creada correctamente.


***Bitácora***: Se crea el documento “Bitácora” para dejar constancia de los problemas detectados, las variables afectadas, las decisiones tomadas y la justificación correspondiente.

In [39]:
registros_bitacora = []

def registrar(problema, variable, decision, justificacion):
    registros_bitacora.append({
        "problema_detectado": problema,
        "variable_afectada": variable,
        "decision_tomada": decision,
        "justificacion": justificacion
    })

### 1. Detección de valores duplicados

Antes de comenzar la limpieza, se revisa si existen registros duplicados en el dataset, ya que podrían generar sesgos al contar dos veces la misma reserva.

En este tipo de datos, dos reservas pueden tener exactamente las mismas características sin ser un error, por lo que solo se consideran duplicados aquellos casos donde todas las columnas coinciden completamente.

Como el dataset no posee un identificador único por reserva, la detección se realiza comparando todas las variables. 

In [40]:
duplicados = df.duplicated().sum()
print(f"Cantidad de registros duplicados: {duplicados}")

Cantidad de registros duplicados: 0


La revisión arrojó **0 duplicados**, por lo que no es necesario aplicar cambios en esta etapa.

---

### 2. Selección de variables

Como parte del análisis inicial, se revisan todas las columnas del dataset para identificar variables que no aporten información relevante al estudio o que no resulten útiles para el análisis de cancelaciones. Este paso permite simplificar el dataset y concentrar el trabajo en las variables que realmente contribuyen a explicar el comportamiento de las reservas.

En nuestro caso, se inspeccionan nuevamente las columnas del DataFrame y se evalúa si alguna corresponde a identificadores irrelevantes, códigos internos, variables redundantes o información que no será utilizada en el análisis. Tras esta revisión, se determina si es necesario eliminar alguna columna o si todas deben conservarse para las etapas posteriores.

In [50]:
lista_variables = df.columns.tolist()
print(f"Listado de variables:\n{lista_variables}")

print("\nVista inicial de la tabla:\n")
df.head()


Listado de variables:
['hotel', 'is_canceled', 'lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'meal', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'reserved_room_type', 'assigned_room_type', 'booking_changes', 'deposit_type', 'agent', 'company', 'days_in_waiting_list', 'customer_type', 'adr', 'required_car_parking_spaces', 'total_of_special_requests']

Vista inicial de la tabla:



,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests
0,Resort Hotel,0,17,2025,February,8,21,0,1,1,...,E,0,No Deposit,NaN,292.0,0,Transient,58.00,1,1
1,Resort Hotel,0,6,2023,November,44,1,2,1,1,...,D,3,No Deposit,240.0,NaN,0,Transient,58.00,0,2
2,Resort Hotel,0,45,2024,April,15,8,0,1,2,...,D,1,No Deposit,240.0,NaN,0,Transient-Party,65.00,0,2
3,City Hotel,0,95,2024,March,11,17,2,3,2,...,A,0,No Deposit,9.0,NaN,0,Transient,73.95,0,1
4,City Hotel,1,277,2024,November,45,7,1,2,2,...,A,0,Non Refund,NaN,NaN,0,Transient,100.00,0,0


**Booking_id**: Se revisa la variable para evaluar si aporta informacion a la hora de realizar el analisis.

In [51]:
# Cantidad de registros en el dataset
registros = len(df)

# Cantidad de valores únicos en booking_id
booking_registros = df['booking_id'].nunique()

print(f"Registros totales en el Dataset: {registros} - Registro unicos en Booking_id: {booking_registros}")



KeyError: 'booking_id'

La variable `booking_id` contiene un valor único para cada registro del dataset.
Al verificar la cantidad de valores únicos, se observa que coincide exactamente con la cantidad total de filas, lo que confirma que funciona únicamente como identificador.
Este tipo de variables no guarda relación con las demás características de la reserva y no aporta información útil para el análisis de cancelaciones. Solamente se trada de un identificador

Por lo tanto, `booking_id` no será utilizada en el análisis exploratorio y puede excluirse de las etapas de diagnóstico y limpieza.

In [ ]:
#Se elimina booking_id
df.drop(columns=['booking_id'], inplace=True)

print("booking_id eliminado correctamente")


Se registra el cambio en la bitácora

In [ ]:
#Se registra el evento
registrar(
    problema="Variable sin aporte analítico",
    variable="booking_id",
    decision="Eliminación de la columna",
    justificacion="booking_id es un identificador único que no se relaciona con las demás variables y no aporta información para explicar cancelaciones."
)


**Arrival Date**: Se revisa la variable para evaluar si aporta informacion a la hora de realizar el analisis.

In [ ]:
df[['arrival_date', 
    'arrival_date_year', 
    'arrival_date_month', 
    'arrival_date_day_of_month',
    'arrival_date_week_number'
]].head()


,arrival_date,arrival_date_year,arrival_date_month,arrival_date_day_of_month,arrival_date_week_number
0,2025-02-21,2025,February,21,8
1,2023-11-01,2023,November,1,44
2,2024-04-08,2024,April,8,15
3,2024-03-17,2024,March,17,11
4,2024-11-07,2024,November,7,45


La columna `arrival_date` contiene la fecha completa de llegada, pero su información ya está desagregada en `arrival_date_year`, `arrival_date_month`, `arrival_date_day_of_month`

Para el analisis, `arrival_date` es redundante, ya que no aporta datos nuevos respecto a las demás variables con datos de fechas que se pueden utilizar para analizar las temporadas del año, mediado del mes de la reserva, años con más o menos reservas o cancelaciones. 

Por este motivo, se procede a eliminarla del DataFrame.


In [ ]:
#Eliminacion de arrival_Date
df.drop(columns=['arrival_date'], inplace=True)

print("arrival_date eliminado correctamente")

Se registra el cambio en la bitácora

In [ ]:
registrar(
    problema="Variable redundante",
    variable="arrival_date",
    decision="Eliminación de la columna",
    justificacion="arrival_date no aporta información nueva en cuanto a los datos de arrivo, todos estos datos ya se encuentran en arrival_date_year, arrival_date_month y arrival_date_day_of_month"
)


**arrival_date_week_number**: Se revisa la variable para evaluar si aporta informacion a la hora de realizar el analisis.

La variable `arrival_date_week_number` indica el número de semana del año, pero su información es redundante comparado a las variables `arrival_date_year`, `arrival_date_month` y `arrival_date_day_of_month`, que ya permiten analizar el comportamiento temporal de las reservas. Es una variable para uso interno de la empresa, que no aporta información adicional relevante para el estudio de cancelaciones, se decide eliminarla del DataFrame.

In [ ]:
#Eliminacion de arrival_date_week_number
df.drop(columns=['arrival_date_week_number'], inplace=True)

print("arrival_date_week_number eliminado correctamente")


Se registra el cambio en la bitácora

In [53]:
registrar(
    problema="Variable redundante",
    variable="arrival_date_week_number",
    decision="Eliminación de la columna",
    justificacion="No aporta información adicional comparado a las variables temporales año, mes y día. Variable para uso interno de la empresa, no aporta informacion para el analisis de los datos."
)


**Agent**: Se revisa la variable para evaluar si aporta informacion a la hora de realizar el analisis.

In [59]:
print(f"Valores repetidos:{df['agent'].value_counts().head(10)}")

print(f"\nValores unicos: {df['agent'].nunique()}")


print(f"\nDatos Faltantes: {df['agent'].isnull().sum()}")


Valores repetidos:agent
9.0      6690
240.0    2896
1.0      1508
7.0       776
14.0      755
6.0       720
250.0     562
241.0     369
28.0      340
8.0       326
Name: count, dtype: int64

Valores unicos: 268

Datos Faltantes: 3489


La variable `agent` presenta *3489* valores faltantes, *268* valores únicos y miles de repeticiones de los mismos códigos, lo que muestra que se trata de un identificador interno del sistema de reservas. No representa una categoría interpretable ni aporta información relevante para explicar cancelaciones. Por lo tanto, se elimina del DataFrame por ser una variable administrativa sin valor analítico.

In [60]:
#Eliminacion de Agent
df.drop(columns=['agent'], inplace=True)

print("agent eliminado correctamente")


agent eliminado correctamente


Se registra el cambio en la bitácora

In [61]:
registrar(
    problema="Código interno sin valor analítico",
    variable="agent",
    decision="Eliminación de la columna",
    justificacion="Presenta muchos faltantes, 268 valores únicos y miles de repeticiones, lo que confirma que es un identificador interno sin aporte al análisis, simplemente una ID para el agente de la reserva."
)


**Company**: Se revisa la variable para evaluar si aporta informacion a la hora de realizar el analisis.

In [62]:
print(f"Valores repetidos:{df['company'].value_counts().head(10)}")

print(f"\nValores unicos: {df['company'].nunique()}")


print(f"\nDatos Faltantes: {df['company'].isnull().sum()}")


Valores repetidos:company
40.0     203
223.0    155
67.0      53
45.0      51
153.0     41
174.0     41
219.0     32
281.0     31
405.0     29
154.0     28
Name: count, dtype: int64

Valores unicos: 218

Datos Faltantes: 23574


La variable `company` presenta *23.574* valores faltantes y *218* valores únicos, además de múltiples repeticiones de los mismos códigos. Esto evidencia que se trata de un identificador interno del sistema de reservas, sin significado analítico ni relación con el comportamiento de cancelaciones. Por lo tanto, se elimina del DataFrame por ser una variable administrativa sin aporte al análisis.

In [ ]:
df.drop(columns=['company'], inplace=True)

print("company eliminado correctamente")


Se registra el cambio en la bitácora

In [64]:
registrar(
    problema="Código interno sin valor analítico",
    variable="company",
    decision="Eliminación de la columna",
    justificacion="Presenta 23.574 faltantes, 218 valores únicos y repeticiones que confirman que es un identificador interno sin aporte al análisis."
)


### Variables eliminadas


- `booking_id`: identificador único sin aporte analítico.

- `arrival_date`: redundante frente a año, mes y día.

- `arrival_date_week_number`: no agrega información relevante para el analisis, es una variable interna para la administración.

- `agent`: código interno con 268 valores únicos y 3489 faltantes, sin significado analítico.

- `company`: código interno con 218 valores únicos y 23.574 faltantes, sin aporte al análisis.

In [66]:
print("Bitacora al momento, con las variables que se eliminaron:\n")
pd.DataFrame(registros_bitacora)


Bitacora al momento, con las variables que se eliminaron:



,problema_detectado,variable_afectada,decision_tomada,justificacion
0,Variable sin aporte analítico (Identificador u...,booking_id,Eliminación de la columna,booking_id es un identificador único que no se...
1,Variable redundante,arrival_date,Eliminación de la columna,arrival_date no aporta información nueva en cu...
2,Variable redundante,arrival_date_week_number,Eliminación de la columna,No aporta información adicional comparado a la...
3,Código interno sin valor analítico,agent,Eliminación de la columna,"Presenta muchos faltantes, 268 valores únicos ..."
4,Código interno sin valor analítico,company,Eliminación de la columna,"Presenta 23.574 faltantes, 218 valores únicos ..."
